# v20 ローカル学習(LightGBM / CatBoost / XGBoost ×3)→ ネイティブ保存

v20 の train.parquet(物理PF×7本の pf_* 特徴 + v9非PF特徴)を読み、GroupKFold(5) by well で
各アルゴ×3設定を **fold毎に学習(early stopping)→ fold モデルを保存**。提出は fold平均×9設定平均。

- koolbox を使わず **素の lightgbm/catboost/xgboost** で学習・joblib保存(提出ノートの依存を最小化)。
- 特徴量 = parquet 全列(well/id/target 除く)。cmp_/bpf_ は v20 に存在しない。
- 保存先: `artifacts_v20/models/{variant}/fold{k}.pkl` + `features.json` + `pp_params.json`。

In [9]:
import json, joblib, warnings, gc, numpy as np, pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
warnings.filterwarnings('ignore')

# ★ PARQUET は create の OUT_PARQUET と一致させる(例: artifacts_v20_2)
ART = Path(r'C:\Users\kosaka256\Documents\rogii_claude\v20\artifacts_v20_3')
PARQUET = ART / 'data' / 'train.parquet'
MODELS = ART / 'models'; MODELS.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
ROW_FRAC = 0.5     # ★メモリ対策: 学習に使う行の割合(1.0=全行)。OOMならさらに下げる(0.3など)。
pp_params = {'alpha': 1.0, 'tau': 85, 'w_pf': 0.09}   # 補正(submitと共有)

df = pd.read_parquet(PARQUET)
features = [c for c in df.columns if c not in {'well', 'id', 'target'}]
(ART).joinpath('features.json').write_text(json.dumps(features))
(ART).joinpath('pp_params.json').write_text(json.dumps(pp_params))
# 行サブサンプル(等間隔)→ float32 → df 解放(ピークRAMを抑える)
if ROW_FRAC < 1.0:
    step = max(1, int(round(1.0 / ROW_FRAC)))
    df = df.iloc[::step].reset_index(drop=True)
y = df['target'].to_numpy(np.float32)
g = df['well'].to_numpy()
X = df[features].astype('float32')
del df; gc.collect()
print('rows:', len(X), ' features:', len(features), ' wells:', pd.unique(g).size, ' ROW_FRAC:', ROW_FRAC)

rows: 1891995  features: 643  wells: 773  ROW_FRAC: 0.5


In [10]:
# v9/v10 と同じパラメータ
lgb_params = [
    dict(boosting_type='gbdt', num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective='regression', verbose=-1,
         n_jobs=-1, device_type='gpu', gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type='gbdt',
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device='gpu'),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type='gbdt',
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device='gpu'),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function='RMSE',
         task_type='GPU', devices='0', od_type='Iter', od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function='RMSE',
         task_type='GPU', devices='0', od_type='Iter', od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function='RMSE',
         task_type='GPU', devices='0', od_type='Iter', od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective='reg:squarederror', eval_metric='rmse', tree_method='hist',
         device='cuda', random_state=0, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective='reg:squarederror', eval_metric='rmse', tree_method='hist',
         device='cuda', random_state=1, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective='reg:squarederror', eval_metric='rmse', tree_method='hist',
         device='cuda', random_state=2, n_jobs=-1, early_stopping_rounds=250),
]
VARIANTS = ([(f'lightgbm-{i+1}', 'lgb', p) for i, p in enumerate(lgb_params)] +
            [(f'catboost-{i+1}', 'cb', p) for i, p in enumerate(cb_params)] +
            [(f'xgboost-{i+1}', 'xgb', p) for i, p in enumerate(xgb_params)])
print('variants:', [v[0] for v in VARIANTS])

variants: ['lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'catboost-1', 'catboost-2', 'catboost-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']


In [11]:
# GroupKFold(5) で fold毎に学習(early stopping)→ foldモデル保存 + OOF収集
# ★ fold{k}.pkl が既にあれば学習せずロード(再開可能・クラッシュしても続きから)
def fit_one(kind, params, Xtr, ytr, Xva, yva):
    if kind == 'lgb':
        m = LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='rmse',
              callbacks=[early_stopping(250), log_evaluation(0)])
    elif kind == 'cb':
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=250, use_best_model=True, verbose=0)
    else:
        m = XGBRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(X, y, groups=g))
oof = {name: np.zeros(len(y), np.float32) for name, _, _ in VARIANTS}
for name, kind, params in VARIANTS:
    d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    for k, (tr, va) in enumerate(folds):
        fp = d / f'fold{k}.pkl'
        if fp.exists():
            m = joblib.load(fp); tag = 'load'           # 学習済み → スキップ
        else:
            m = fit_one(kind, params, X.iloc[tr], y[tr], X.iloc[va], y[va])
            joblib.dump(m, fp); tag = 'fit'
        oof[name][va] = m.predict(X.iloc[va])
        print(f'{name} fold{k} [{tag}]: rmse={root_mean_squared_error(y[va], oof[name][va]):.4f}', flush=True)
        del m; gc.collect()
    print(f'== {name} OOF rmse={root_mean_squared_error(y, oof[name]):.4f} ==', flush=True)

oof_mean = np.mean([oof[n] for n, _, _ in VARIANTS], axis=0)
print(f'\nFinal OOF RMSE (9-model mean, target上): {root_mean_squared_error(y, oof_mean):.4f}')
print('models saved under:', MODELS)
print('→ artifacts_v20(models/ + features.json + pp_params.json)+ pf_bank.json + v9ノートを Kaggle にアップ')

Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[671]	valid_0's rmse: 6.67025	valid_0's l2: 44.4923
lightgbm-1 fold0 [fit]: rmse=6.6703
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[623]	valid_0's rmse: 7.129	valid_0's l2: 50.8227
lightgbm-1 fold1 [fit]: rmse=7.1290
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[166]	valid_0's rmse: 7.6453	valid_0's l2: 58.4505
lightgbm-1 fold2 [fit]: rmse=7.6453
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[64]	valid_0's rmse: 8.28467	valid_0's l2: 68.6357
lightgbm-1 fold3 [fit]: rmse=8.2847
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1124]	valid_0's rmse: 8.06222	valid_0's l2: 64.9993
lightgbm-1 fold4 [fit]: rmse=8.0622
== lightgbm-1 OOF rmse=7.5817 ==
Training until validation scores don't improv

KeyboardInterrupt: 